In [1]:
import pyranges as pr
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
BASE = Path("../data/alzheimer-lrseq")
BASE_GTF = Path("../.data")

SAMPLES = pd.DataFrame([
    {"sample_id": "alzheimer",  "condition": "Alzheimer", "age": 90},
    {"sample_id": "alzheimer2", "condition": "Alzheimer", "age": 86},
    {"sample_id": "healthy",    "condition": "Healthy",   "age": 90},
    {"sample_id": "healthy2",   "condition": "Healthy",   "age": 85},
])

MOD_TYPE_MAP = {
    "a": "m6A", "m": "m5C",
    "17802": "Pseudouridine", "17596": "Inosine",
    "19227": "Nm", "19228": "Nm", "19229": "Nm", "69426": "Nm",
}

BED_COLS = ["chrom","start","end","mod_code","score","strand",
            "thick_start","thick_end","color","coverage","mod_freq",
            "mod_count","unmod_count","v14","v15","v16","v17","v18"]

In [13]:
def load_exons(sample_id):
    gtf_path = BASE_GTF / "stringtie" / sample_id / "transcripts_filtered.gtf"
    gtf = pr.read_gtf(str(gtf_path))
    df = gtf[gtf.Feature == "exon"].df
    df = df[df["Strand"].isin(["+", "-"])].copy()
    df = df.reset_index(drop=True)
    return pr.PyRanges(df[["Chromosome", "Start", "End", "Strand", "transcript_id"]])

exons_by_sample = {}
for sid in SAMPLES.sample_id:
    exons_by_sample[sid] = load_exons(sid)
    print(f"  {len(exons_by_sample[sid])} exons")

  267139 exons
  435652 exons
  494993 exons
  364018 exons


In [10]:
def load_bed(sample_id):
    bed_dir = BASE / "modifications" / sample_id / "bed"
    dfs = []
    for f in sorted(bed_dir.glob("*.bed.gz")):
        df = pd.read_csv(
            f, sep = "\t", header = None,
            names = BED_COLS, usecols = ["chrom","start","end","mod_code","mod_freq","coverage","strand"]
        )
        dfs.append(df)
    bed = pd.concat(dfs, ignore_index = True)
    bed["mod_type"] = bed["mod_code"].astype(str).map(MOD_TYPE_MAP).fillna(bed["mod_code"].astype(str))
    return bed

bed_by_sample = {}
for sid in SAMPLES.sample_id:
    bed_by_sample[sid] = load_bed(sid)
    print(f"  {len(bed_by_sample[sid]):,} modification sites")
    print(bed_by_sample[sid].mod_type.value_counts().to_string())

  348,274 modification sites
mod_type
m6A              197201
m5C               58487
Nm                56302
Inosine           19622
Pseudouridine     16662
  696,986 modification sites
mod_type
m6A              378117
m5C              122510
Nm               116125
Inosine           46663
Pseudouridine     33571
  661,266 modification sites
mod_type
m6A              331682
m5C              133123
Nm               115724
Inosine           44176
Pseudouridine     36561
  373,790 modification sites
mod_type
m6A              197990
m5C               74331
Nm                59408
Inosine           21226
Pseudouridine     20835


In [15]:
def overlap_sample(sample_id):
    exons_pr = exons_by_sample[sample_id]

    bed = bed_by_sample[sample_id]
    bed_pr = pr.PyRanges(bed.rename(columns={
        "chrom": "Chromosome", "start": "Start", "end": "End", "strand": "Strand"
    }))

    joined = exons_pr.join(bed_pr, strandedness=False)
    df = joined.df
    df = df[df["Strand"] == df["Strand_b"]]
    return df[["transcript_id", "mod_type", "mod_freq", "coverage"]]

overlaps = {}
for sid in SAMPLES.sample_id:
    print(f"Overlapping: {sid}")
    overlaps[sid] = overlap_sample(sid)
    print(f"  {len(overlaps[sid]):,} modification-exon hits")

Overlapping: alzheimer


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


  748,353 modification-exon hits
Overlapping: alzheimer2


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


  1,800,980 modification-exon hits
Overlapping: healthy


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


  1,873,962 modification-exon hits
Overlapping: healthy2
  909,973 modification-exon hits


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.
